In [3]:
import numpy as np
import pandas as pd
import xarray as xr
from functools import reduce
import matplotlib.pyplot as plt
from idlpy import *
from scipy import stats
import statsmodels.api as sm
from scipy.stats import percentileofscore
import scipy
import os
import cartopy.crs as ccrs

In [5]:
'''
Read time slice netcdfs of HadGEM3 and write single files in /nesi/project/niwa00015/queenle/data/
'''

# land/sea mask
HadGEM_land_mask = xr.open_dataset('/nesi/project/niwa00015/models/C20C/MOHC/HadGEM3-A-N216/All-Hist/est1/v1-0/fx/atmos/sftlf/r0i0p0/sftlf_fx_HadGEM3-A-N216_All-Hist_est1_v1-0_r0i0p0_000000-000000.nc')
HadGEM_land_mask = HadGEM_land_mask.sel(lon = slice(165,180),lat=slice(-48,-33))

seasons_a = ['autumn','winter','spring','summer']

ens_list = ['r1i1p1','r1i1p11','r1i1p13','r1i1p15','r1i1p3','r1i1p5','r1i1p7','r1i1p9',
            'r1i1p10','r1i1p12','r1i1p14','r1i1p2','r1i1p4','r1i1p6','r1i1p8']

slice_list = ['196001-196912','197001-197912','198001-198912','199001-199912','200001-200912',
              '201001-201312','201401-201512']

path_info = {'all':['/nesi/project/niwa00015/models/C20C/MOHC/HadGEM3-A-N216/All-Hist/est1/v1-0/mon/atmos/tasmax/',
                    'tasmax_Amon_HadGEM3-A-N216_All-Hist_est1_v1-0_'],
             'nat':['/nesi/project/niwa00015/models/C20C/MOHC/HadGEM3-A-N216/Nat-Hist/CMIP5-est1/v1-0/mon/atmos/tasmax/',
                    'tasmax_Amon_HadGEM3-A-N216_Nat-Hist_CMIP5-est1_v1-0_']
            }
'''
for scen in ['all','nat']:
    
    for ens_mem in ens_list:
        
        in_directory = path_info[scen][0] + ens_mem + '/'
        out_directory = "/nesi/project/niwa00015/queenle/data/HadGEM3/tasmax/" + scen + "/" + path_info[scen][1] + ens_mem + "_1960_2015.nc"        
                
        os.system('cdo mergetime ' + in_directory + '* ' + out_directory)
'''

'\nfor scen in [\'all\',\'nat\']:\n    \n    for ens_mem in ens_list:\n        \n        in_directory = path_info[scen][0] + ens_mem + \'/\'\n        out_directory = "/nesi/project/niwa00015/queenle/data/HadGEM3/tasmax/" + scen + "/" + path_info[scen][1] + ens_mem + "_1960_2015.nc"        \n                \n        os.system(\'cdo mergetime \' + in_directory + \'* \' + out_directory)\n'

In [23]:
def q_map_to_gaussian(da, dist):
    
    # create container for corrected values
    gaussian = []
    
    for lat in da.lat:
        for lon in da.lon:

            ts = da.sel(lat=lat,lon=lon).values
            full_dist = dist.sel(lat=lat,lon=lon).values.flatten()
            
            if np.isnan(np.sum(ts)):
                corrected_ts = [np.nan for i in range(len(ts))]

            else:
                corrected_ts = []
                for val in ts:
                    
                    # find percentile of value within the distribution of observations
                    percentile = percentileofscore(full_dist, val, kind= 'mean')
                    
                    if percentile == 100 or percentile == 0:
                        print('issue')

                    # replace with corresponding percentile value in norm distribution
                    corrected_ts.append(scipy.stats.norm.ppf(percentile/100, loc=0, scale=1))

            gaussian.append(corrected_ts)

    gaussian = np.array(gaussian).reshape((len(da.lat),len(da.lon),len(da.time)))
    
    gaussian_da = xr.DataArray(data = gaussian,
                               dims = ('lat', 'lon','time'),
                               coords = {'lat':da.lat.values,
                                         'lon':da.lon.values,
                                         'time':da.time.values
                                        }
                               )
    return(gaussian_da)


In [18]:
'''
Creating dataarrays of FULL All-forcing distributions (concatenate all ensemble members over time) for seasonal pr
'''

seasons_a = ['autumn','winter','spring','summer']
path = '/nesi/project/niwa00015/queenle/data/HadGEM3/tasmax/'
file_name_setup = {'all':['tasmax_Amon_HadGEM3-A-N216_All-Hist_est1_v1-0_','_1960_2015.nc'],
                   'nat':['tasmax_Amon_HadGEM3-A-N216_Nat-Hist_CMIP5-est1_v1-0_','_1960_2015.nc']}

def preprocess_ds(ds):
    
    processed_ds = ds.sel(lon = slice(165,180),lat=slice(-48,-33))
    processed_ds = processed_ds.tasmax.where(HadGEM_land_mask.sftlf > 0)

    processed_ds = processed_ds.resample(time='QS-DEC').mean()
    processed_ds = processed_ds.sel(time=slice('1960-12-01','2015-09-01'))
    
    return(processed_ds)
        
full_ensembles = {}
for scen in ['all','nat']:

    da_list = []
    for mem_name in ens_list:
        file_path = path +scen + '/' + file_name_setup[scen][0]+mem_name+file_name_setup[scen][1]
        ens_mem = xr.open_dataset(file_path)
    
        processed_ds = preprocess_ds(ens_mem)

        da_list.append(processed_ds)

    full_ensemble = xr.concat(da_list,dim="member")
    full_ensemble['member'] = ens_list
    
    full_ensembles[scen] = full_ensemble


In [25]:
full_ensembles['all']

<xarray.DataArray 'tasmax' (member: 15, time: 220, height: 1, lat: 27, lon: 18)>
array([[[[[      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan, 286.3522 , 286.81516, ...,       nan,       nan,
                 nan],
          ...,
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan]]],


        [[[      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan, 285.4705 , 284.99118, ...,       nan,       nan,
...
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan]]],


        [[[      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan, 283.43307, 283.43906, ...,       nan,       nan,
                 nan],
          ...,
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan],
          [      nan,       nan,       nan, ...,       nan,       nan,
                 nan]]]]], dtype=float32)
Coordinates:
  * time     (time) object 1960-12-01 00:00:00 ... 2015-09-01 00:00:00
  * lon      (lon) float32 165.4 166.2 167.1 167.9 ... 177.1 177.9 178.8 179.6
  * lat      (lat) float32 -47.5 -46.94 -46.39 -45.83 ... -34.17 -33.61 -33.06
  * height   (height) float32 2.0
  * member   (member) <U7 'r1i1p1' 'r1i1p11' 'r1i1p13' ... 'r1i1p6' 'r1i1p8'

In [27]:
'''
gaussian-ifying annual and seasonal pr for each ensemble member
'''

seasons_a = ['autumn','winter','spring','summer']
path = '/nesi/project/niwa00015/queenle/data/HadGEM3/tasmax/'
file_name_setup = {'all':['tasmax_Amon_HadGEM3-A-N216_All-Hist_est1_v1-0_','_1960_2015.nc'],
                   'nat':['tasmax_Amon_HadGEM3-A-N216_Nat-Hist_CMIP5-est1_v1-0_','_1960_2015.nc']}


for scen in ['all','nat']:
    
    ens_ds = full_ensembles[scen]

    for mem_name in ens_list:
        
        file_path = path +scen + '/' + file_name_setup[scen][0]+mem_name+file_name_setup[scen][1]
        ens_mem = xr.open_dataset(file_path)
    
        processed_ds = preprocess_ds(ens_mem)
        
        for i,s in enumerate([3,6,9,12]):
            
            ens_season = ens_ds.sel(time=ens_ds.time.dt.month.isin([s]))
            ens_mem_season = processed_ds.sel(time=processed_ds.time.dt.month.isin([s]))

            ens_mem_season_gaussian = q_map_to_gaussian(ens_mem_season,ens_season)
            ens_mem_season_gaussian.to_netcdf("/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/gaussian/tasmax/seasonal/HadGEM3_"+scen+"_"+seasons_a[i]+"_tasmax_gaussian_"+mem_name+".nc")
            

In [38]:
'''
Create ensemble means based on gaussian ensemble members for annual, seasonal pr
'''

seasons_a = ['autumn','winter','spring','summer']
path = "/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/gaussian/tasmax/seasonal/"
    
for scen in ['all','nat']:

    file_name = ["HadGEM3_"+scen+"_","_tasmax_gaussian_",".nc"]
    
    for i,s in enumerate([3,6,9,12]):

        da_list = []
        for mem_name in ens_list:
            file_path = path + file_name[0]+seasons_a[i]+file_name[1]+mem_name+file_name[2]
            ens_mem_season = xr.open_dataarray(file_path)

            da_list.append(ens_mem_season)

        ens_ds = xr.concat(da_list,dim='member')
        ens_ds['member']=ens_list
        
        # full ensemble mean
        ens_ds.mean('member').to_netcdf(path+'means/'+file_name[0]+seasons_a[i]+file_name[1]+"ensMean"+file_name[2])
        
        # means without members
        for mem_name in ens_list:
            ens_ds.where(ens_ds.member != mem_name).mean('member').to_netcdf(path+'means/'+file_name[0]+seasons_a[i]+file_name[1]+"mean_sub_"+mem_name+'_'+file_name[2])


In [6]:
'''
Normalize by removing local mean from each ensemble member
'''

for scen in ['all','nat']:
    print(scen)

    if scen == 'all':
        path = '/nesi/project/niwa00015/queenle/data/HadGEM3/tasmax/all/'
        file_name_setup = ['tasmax_Amon_HadGEM3-A-N216_All-Hist_est1_v1-0_','_1960_2015.nc']
    if scen == 'nat':
        path = '/nesi/project/niwa00015/queenle/data/HadGEM3/tasmax/nat/'
        file_name_setup = ['tasmax_Amon_HadGEM3-A-N216_Nat-Hist_CMIP5-est1_v1-0_','_1960_2015.nc']
        
    for mem_name in ens_list:
        
        file_path = path + file_name_setup[0]+mem_name+file_name_setup[1]
        ens_mem = xr.open_dataset(file_path)
        
        ens_mem = ens_mem.sel(lon = slice(165,180),lat=slice(-48,-33))
        ens_mem = ens_mem.tasmax.where(HadGEM_land_mask.sftlf > 0)

        ens_mem_seasonal = ens_mem.resample(time='QS-DEC').mean()
        ens_mem_seasonal = ens_mem_seasonal.sel(time=slice('1960-12-01','2015-09-01'))
        
        ens_mem.close()
        for i,s in enumerate([3,6,9,12]):
            
            ens_mem_season = ens_mem_seasonal.sel(time=ens_mem_seasonal.time.dt.month.isin([s]))

            anom = ens_mem_season - ens_mem_season.mean('time')
            anom.to_netcdf("/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/lmr/tasmax/seasonal/HadGEM3_"+scen+"_"+seasons_a[i]+"_tasmax_LMR_"+mem_name+".nc")

            ens_mem_season.close()
            anom.close()
            
        ens_mem_seasonal.close()

all
nat


In [13]:
'''
Create ensemble means based on gaussian ensemble members for annual, seasonal pr
'''

seasons_a = ['autumn','winter','spring','summer']

for scen in ['all','nat']:
    
    # SEASONAL
    path = "/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/lmr/tasmax/seasonal/"
    file_name = ["HadGEM3_"+scen+"_","_tasmax_LMR_",".nc"]
    
    for i,s in enumerate([3,6,9,12]):

        da_list = []
        for mem_name in ens_list:
            file_path = path + file_name[0]+seasons_a[i]+file_name[1]+mem_name+file_name[2]
            ens_mem_season = xr.open_dataarray(file_path)

            da_list.append(ens_mem_season)

        season_ens_mean = sum(da_list)/len(da_list)
        season_ens_mean.to_netcdf(path+file_name[0]+seasons_a[i]+file_name[1]+"ensMean"+file_name[2])
